# Sectoral aggregation — OECD & WiNDC

Loads the raw OECD ICIO and WiNDC tables, applies the proposed sectoral mapping, aggregates both onto the common sector set, and runs the aggregation diagnostics/tests. Outputs (aggregated tables) are written to disk for the downstream notebooks.

# Librairies

In [ ]:
from paths import ROOT
import pandas as pd
import numpy as np
from pathlib import Path
import os

# Agregation of the OECD & WiNDC tables sectors

## Loading tables

In [ ]:
version_windc = 'v3.1_RAS'

In [ ]:
# ===========================================================================
# Paths
# ===========================================================================
CONC_PATH   = ROOT / "data/raw/correspondence/windc_oecd_concordance_handmade_cleaned_correspondence.csv"
OECD_ROOT   = ROOT / "data/interim/IOT/OCDE ICIO"
# WiNDC build to aggregate: grav_fric_v3.1_RAS = current production build
# (v1.2-A md0 chain + column RAS + VA convention; built by
#  ../build_intraUS_IOT/v3_construction.ipynb for ALL years).
WINDC_ROOT  = ROOT / f"data/interim/IOT/IOT_USA/grav_fric_{version_windc}"
OUT_OECD    = OECD_ROOT.parent / "OCDE ICIO aggregated"
OUT_WINDC   = WINDC_ROOT.parent / f"grav_fric_{version_windc}_aggregated"

OUT_OECD.mkdir(exist_ok=True)
OUT_WINDC.mkdir(exist_ok=True)

# ── Year selection ────────────────────────────────────────────────────────────
# PRODUCTION DEFAULT = None -> process EVERY year on both sides. This is required
# for a homogeneous sector scheme across the whole series: a partial run with
# VERSION='' overwrites only some years of the BASE aggregated OECD dir with the
# new mapping, leaving the series mixed (this happened for 2017: the base dir held
# 2017 with the 37-sector mapping and every other year with the legacy 39-sector
# one, and the canonical nested series inherited the inconsistency).
# YEARS = ['2017'] is only for quick single-year tests kept OUT of the base dir
# (then also set a non-empty VERSION).
YEARS   = None
# Output sub-version. '' writes to the BASE aggregated dir that analysis_comparison
# reads (its version=None); set a name (e.g. 'osv+whtrade') to keep variants separate.
VERSION = ''


## Proposed sectoral mapping

In [ ]:
# ===========================================================================
# 1. Build sector mappings from concordance + apply manual fixes
# ===========================================================================
conc = pd.read_csv(CONC_PATH)
print(conc[conc['windc_sector']=="rnt"])

In [ ]:
oecd_to_proposed  = (conc.drop_duplicates("oecd_code")
                        .set_index("oecd_code")["proposed sector"].to_dict())
windc_to_proposed = (conc.drop_duplicates("windc_sector")
                        .set_index("windc_sector")["proposed sector"].to_dict())

# Fix 1 (rnt): OECD code N bundles 561 (admin) + 532RL (rental) into
# "Administrative and support service activities". Concordance put WinDC `rnt`
# in "Other service activities" — realign to OECD.
windc_to_proposed["rnt"] = "Administrative and support service activities"

# Fix 2 (mmf): OECD C31T33 = furniture + other manufacturing + repair. WinDC
# split = fpd (337/furniture) + mmf (339/misc manuf). Concordance put mmf
# into machinery — move it to furniture so WD furniture matches OECD C31T33.
windc_to_proposed["mmf"] = "Manufacture of furniture"

# Fix 3 (education / public admin): BEA convention splits public education
# into government sectors, so WinDC `edu` is private-only (1/4 of OECD P).
# Merge Education + Public admin into one sector on both sides.
_merged = "Education and public administration"
for d in (windc_to_proposed, oecd_to_proposed):
    for k, v in list(d.items()):
        if v in ("Education", "Public administration and defence"):
            d[k] = _merged

# Fix 4 (osv): NAICS 811 (repair) leaks from BEA osv into OECD G (auto repair).
# Merge "Other service activities" with "Wholesale and retail trade" so the
# leakage is contained within one bucket and OUT correspondence is exact.

# Fix 5 (fen / sle): OECD H53 (postal & courier) bundles USPS + private couriers,
# but WiNDC `fen` is USPS only (~$71bn) while the private couriers are booked under
# `otr`/`wrh` (warehousing / transport support). Folding BOTH `fen` and OECD H53
# into the warehousing bucket cancels the two opposite-sign gaps (GFE -79,
# warehousing +119 -> +39 Bn$) and removes the GFE orphan bucket. Validated on
# v3.1 2017: L1(output shares) 0.098 -> 0.094.  `sle` is LEFT in public
# administration on purpose: every alternative (utilities, transport, own bucket)
# worsens the fit, because its +$107bn offsets the education shortfall in the
# merged Education+PubAdmin bucket.
_warehouse = windc_to_proposed["wrh"]   # "Warehousing and support activities for transportation"
windc_to_proposed["fen"] = _warehouse
oecd_to_proposed["H53"]  = _warehouse

In [ ]:
# display the sectors contained in `conc` (defined in a previous cell)
proposed = sorted(conc["proposed sector"].dropna().unique())
windc = sorted(conc["windc_sector"].dropna().unique())

print(f"{len(proposed)} proposed sectors:")
for s in proposed:
    print(s)

print(f"\n{len(windc)} WinDC sectors:")
for s in windc:
    print(s)

In [ ]:
print(f"OECD  : {len(oecd_to_proposed)} codes  → {len(set(oecd_to_proposed.values()))} proposed sectors")
print(f"WinDC : {len(windc_to_proposed)} sectors → {len(set(windc_to_proposed.values()))} proposed sectors")

## Mapping validation — cell-by-cell (heatmap) test

Scores every combination of the candidate mapping tweaks by how well the two
tables agree **cell by cell**: the L1 distance between the *normalized*
`Z[USA×USA]` intermediate-flow blocks of WiNDC (v3.1, with `fen`/`sle` kept) and
OECD ICIO. Output-share L1 is reported alongside for reference.

In [ ]:
# ── Mapping validation: cell-by-cell (heatmap) test on Z[USA x USA] ───────────
# Screening diagnostic behind the choice of sector mapping: it scores every candidate
# tweak against the OECD US block on the PRE-RAS build grav_fric_v3.1. That build is a
# screening artefact, not a step of the delivered chain, so the cell is skipped when it
# is absent -- the production run only needs grav_fric_v3.1_RAS.
_WINDC_TEST_PATH = WINDC_ROOT.parent / "grav_fric_v3.1" / "IOT_2017.npz"
if not _WINDC_TEST_PATH.exists():
    print(f"skipped: {_WINDC_TEST_PATH.relative_to(ROOT)} not present "
          f"(pre-RAS screening build, not required for the delivered series)")
else:
    # ── Mapping validation: cell-by-cell (heatmap) test on Z[USA x USA] ───────────
    # WiNDC (v3.1) vs OECD ICIO US block, every combination of the candidate tweaks.
    # Metric = L1 between the two NORMALIZED Z[USA x USA] matrices (structure), with
    # output-share L1 alongside. Lower = closer. Uses `conc`, OECD_ROOT, WINDC_ROOT.
    import glob, itertools
    import numpy as np, pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.colors import LogNorm

    TEST_YEAR  = "2017"
    WINDC_TEST = WINDC_ROOT.parent / "grav_fric_v3.1" / f"IOT_{TEST_YEAR}.npz"   # fen/sle kept
    FDc = {"HFCE", "NPISH", "GGFC", "GFCF", "INVNT", "DPABR"}

    ob0 = conc.drop_duplicates("oecd_code").set_index("oecd_code")["proposed sector"].to_dict()
    wb0 = conc.drop_duplicates("windc_sector").set_index("windc_sector")["proposed sector"].to_dict()

    # OECD USA x USA intermediate block (by OECD code) + USA output per code
    ofile = next(f for folder in sorted(OECD_ROOT.iterdir()) if folder.is_dir()
                 for f in folder.glob(f"{TEST_YEAR}_*.parquet"))
    odf = pd.read_parquet(ofile)
    orows = [r for r in odf.index if isinstance(r, str) and r.startswith("USA_") and r.split("_", 1)[1] not in FDc]
    ocols = [c for c in odf.columns if c.startswith("USA_") and c.split("_", 1)[1] not in FDc]
    Zo = odf.loc[orows, ocols].astype(float)
    Zo.index   = [r.split("_", 1)[1] for r in orows]
    Zo.columns = [c.split("_", 1)[1] for c in ocols]
    oo = {c: float(odf.loc["OUT"]["USA_" + c]) for c in Zo.columns}

    # WiNDC USA x USA block (sector x sector, state-summed, Bn$ -> M$) + output
    nz = np.load(WINDC_TEST, allow_pickle=True)
    secs = [str(s) for s in nz["sectors"]]; n_r = len(nz["regions"]); S = len(secs)
    Zw = pd.DataFrame(nz["Z"].reshape(n_r, S, n_r, S).sum(axis=(0, 2)) * 1000.0, index=secs, columns=secs)
    wo = dict(zip(secs, (nz["Z"].sum(1) + nz["F"].sum(1) + nz["EX"]).reshape(n_r, S).sum(0) * 1000.0))

    B_MERGE = "Education and public administration"
    B_TRADE = "Other services + wholesale/retail"

    def _maps(mmf, rnt, edu, trade, fen, sle):
        o, w = dict(ob0), dict(wb0)
        w["mmf"] = {"mach": wb0["mch"], "furn": wb0["fpd"]}[mmf]
        w["rnt"] = {"osv": wb0["osv"], "admin": wb0["adm"], "re": wb0["hou"]}[rnt]
        if edu:
            for d in (o, w):
                for k, v in list(d.items()):
                    if v in (wb0["edu"], wb0["fdd"]): d[k] = B_MERGE
        if trade:
            for d in (o, w):
                for k, v in list(d.items()):
                    if v in (wb0["osv"], wb0["wht"]): d[k] = B_TRADE
        if fen == "wh":   w["fen"] = wb0["otr"]; o["H53"] = wb0["otr"]
        elif fen == "pa": t = B_MERGE if edu else wb0["fdd"]; w["fen"] = t; o["H53"] = t
        if sle == "util": w["sle"] = wb0["uti"]
        elif sle == "land": w["sle"] = wb0["pip"]
        return o, w

    def _am(df, mp):
        d = df.copy()
        d.index   = [mp.get(i, i) for i in d.index]
        d.columns = [mp.get(c, c) for c in d.columns]
        return d.groupby(level=0).sum().T.groupby(level=0).sum().T

    def _scores(o, w):
        Mo, Mw = _am(Zo, o), _am(Zw, w)
        k = sorted(set(Mo.index) | set(Mw.index))
        Mo = Mo.reindex(index=k, columns=k, fill_value=0).values
        Mw = Mw.reindex(index=k, columns=k, fill_value=0).values
        L1m = np.abs(Mo / Mo.sum() - Mw / Mw.sum()).sum()
        vo = pd.Series(oo); vo.index = [o.get(i, i) for i in vo.index]; vo = vo.groupby(level=0).sum()
        vw = pd.Series(wo); vw.index = [w.get(i, i) for i in vw.index]; vw = vw.groupby(level=0).sum()
        kk = sorted(set(vo.index) | set(vw.index))
        vo = vo.reindex(kk, fill_value=0).values; vw = vw.reindex(kk, fill_value=0).values
        return L1m, np.abs(vo / vo.sum() - vw / vw.sum()).sum()

    LEVERS = dict(mmf=["mach", "furn"], rnt=["osv", "admin", "re"], edu=[False, True],
                  trade=[False, True], fen=["gfe", "wh", "pa"], sle=["pa", "util", "land"])
    recs = []
    for combo in itertools.product(*LEVERS.values()):
        cfg = dict(zip(LEVERS, combo)); l1m, l1o = _scores(*_maps(**cfg))
        recs.append({**cfg, "L1_matrix": round(l1m, 4), "L1_output": round(l1o, 4)})
    grid = pd.DataFrame(recs).sort_values("L1_matrix").reset_index(drop=True)
    pd.set_option("display.width", 220)
    print(f"{len(grid)} variants — TOP 8 by cell-by-cell (matrix) L1:")
    print(grid.head(8).to_string(index=False))
    print("\nrank agreement matrix vs output L1 (Spearman): "
          f"{grid['L1_matrix'].corr(grid['L1_output'], method='spearman'):.3f}")

    BEST = grid.iloc[0][list(LEVERS)].to_dict()
    base = grid["L1_matrix"].min()
    print("\nper-lever sensitivity from the optimum (delta matrix-L1, + = worse):")
    for lev, opts in LEVERS.items():
        for alt in opts:
            if alt == BEST[lev]: continue
            cfg = dict(BEST); cfg[lev] = alt
            print(f"  {lev:6s} {str(BEST[lev]):5s} -> {str(alt):5s} : {_scores(*_maps(**cfg))[0] - base:+.4f}")
    print("\nADOPTED (optimum):", BEST)

    # ── Heatmap of the optimum: normalized Z structure, WiNDC vs OECD + difference ──
    o, w = _maps(**BEST); Mo, Mw = _am(Zo, o), _am(Zw, w)
    k = sorted(set(Mo.index) & set(Mw.index))
    Mo = Mo.reindex(index=k, columns=k, fill_value=0).values; Mw = Mw.reindex(index=k, columns=k, fill_value=0).values
    Mo_n, Mw_n = Mo / Mo.sum(), Mw / Mw.sum()
    fig, ax = plt.subplots(1, 3, figsize=(22, 7))
    vmax = max(Mw_n.max(), Mo_n.max())
    for a, (Mn, t) in zip(ax[:2], [(Mw_n, "WiNDC"), (Mo_n, "OECD")]):
        im = a.imshow(np.clip(Mn, 1e-6, None), cmap="viridis", norm=LogNorm(vmin=1e-6, vmax=vmax), aspect="auto")
        a.set_title(f"{t} — normalized Z[USA x USA] (log)")
        a.set_xticks(range(len(k))); a.set_xticklabels([s[:10] for s in k], rotation=90, fontsize=5)
        a.set_yticks(range(len(k))); a.set_yticklabels([s[:10] for s in k], fontsize=5)
        plt.colorbar(im, ax=a, shrink=0.7)
    diff = Mw_n - Mo_n; dmax = np.abs(diff).max()
    im = ax[2].imshow(diff, cmap="RdBu_r", vmin=-dmax, vmax=dmax, aspect="auto")
    ax[2].set_title("WiNDC - OECD (normalized, common sectors)")
    ax[2].set_xticks(range(len(k))); ax[2].set_xticklabels([s[:10] for s in k], rotation=90, fontsize=5)
    ax[2].set_yticks(range(len(k))); ax[2].set_yticklabels([s[:10] for s in k], fontsize=5)
    plt.colorbar(im, ax=ax[2], shrink=0.7)
    plt.tight_layout(); plt.show()

## Decision — sectoral aggregation (validated cell-by-cell)

Every lever below **reduces** the cell-by-cell distance between the two `Z[USA×USA]`
blocks (and the two metrics rank the 216 variants almost identically, Spearman ≈ 0.86),
so all are adopted:

| tweak | what | why | Δ matrix-L1 if reverted |
|---|---|---|---|
| **rnt → Admin & support** | rental/leasing (532RL) → OECD N | OECD N bundles admin + rental | +0.023 |
| **Education + Public admin merged** | one bucket both sides | BEA books public education in government → WiNDC `edu` ≈ ¼ of OECD P | +0.026 |
| **mmf → Furniture/other-manuf** | misc manufacturing → OECD C31T33 | C31T33 = furniture + other manufacturing; C28 = machinery only | +0.009 |
| **fen + OECD H53 → Warehousing/transport** | USPS + postal/couriers together | OECD H53 = postal + private couriers; WiNDC books couriers in `otr`/`wrh`, `fen` = USPS only | +0.012 |
| **Other services + Wholesale/retail merged** | one bucket both sides | NAICS 811 (repair) leaks from BEA `osv` into OECD G | +0.005 |

`sle` is **kept in Public administration**: every alternative (utilities, transport, own
bucket) *worsens* the structure (its ≈$107 bn partly offset the education shortfall in the
merged government bucket).

The next cell implements this final mapping. It **supersedes the manual fixes above** by
rebuilding both dictionaries from the concordance with all five tweaks applied.

In [ ]:
# ── FINAL sectoral aggregation mapping (everything that improves the ──────────
# cell-by-cell USA correspondence; see the validation test above). This cell is
# the authoritative mapping consumed by the OECD/WiNDC aggregation below.
oecd_to_proposed  = conc.drop_duplicates("oecd_code").set_index("oecd_code")["proposed sector"].to_dict()
windc_to_proposed = conc.drop_duplicates("windc_sector").set_index("windc_sector")["proposed sector"].to_dict()

# (1) rnt -> Administrative & support  (OECD N bundles rental 532RL)
windc_to_proposed["rnt"] = "Administrative and support service activities"

# (2) mmf -> Furniture/other-manuf  (OECD C31T33 = furniture + misc manufacturing)
windc_to_proposed["mmf"] = "Manufacture of furniture"

# (3) Education + Public administration merged  (BEA books public education in gov)
for d in (windc_to_proposed, oecd_to_proposed):
    for k, v in list(d.items()):
        if v in ("Education", "Public administration and defence"):
            d[k] = "Education and public administration"

# (4) Other services + Wholesale/retail merged  (NAICS 811 repair leaks into OECD G)
for d in (windc_to_proposed, oecd_to_proposed):
    for k, v in list(d.items()):
        if v in ("Other service activities", "Wholesale and retail trade, stores"):
            d[k] = "Other services + wholesale/retail"

# (5) fen + OECD H53 -> Warehousing/transport support  (OECD postal bundles private
#     couriers WiNDC books in otr/wrh; fen = USPS only). sle stays in public admin.
_warehouse = windc_to_proposed["wrh"]   # "Warehousing and support activities for transportation"
windc_to_proposed["fen"] = _warehouse
oecd_to_proposed["H53"]  = _warehouse

print(f"final mapping: OECD {len(set(oecd_to_proposed.values()))} | "
      f"WiNDC {len(set(windc_to_proposed.values()))} proposed sectors")

## Agregation of OECD

### Function

In [ ]:
# ===========================================================================
# 2. OECD aggregation helpers
# ===========================================================================
EXTRA_LABELS = {"TLS", "VA", "OUT"}  # non-sector rows/cols in OECD tables

def parse_oecd_label(label):
    """Split 'AGO_C17_18' → ('AGO', 'C17_18'); special rows → (label, None)."""
    if label in EXTRA_LABELS or "_" not in label:
        return label, None
    return label[:3], label[4:]   # country = first 3 chars; sector = rest after '_'

def _map_label(label, mapping):
    cty, sec = parse_oecd_label(label)
    if sec is None:
        return label
    return f"{cty}_{mapping.get(sec, sec)}"   # unmapped sectors keep their code


In [ ]:
def aggregate_oecd(df):
    """Aggregate one OECD MRIO DataFrame to the proposed-sector classification."""
    
    # ---- rows ----
    new_row_labels = [_map_label(r, oecd_to_proposed) for r in df.index]
    df = df.copy()
    df.index = new_row_labels
    df_agg = df.groupby(level=0).sum()

    # ---- columns ----
    new_col_labels = [_map_label(c, oecd_to_proposed) for c in df_agg.columns]
    df_agg.columns = new_col_labels
    df_agg = df_agg.T.groupby(level=0).sum().T

    return df_agg

In [ ]:
# ===========================================================================
# 4. Process OECD years  (filtered by YEARS; VERSION='' -> base dir)
# ===========================================================================
for folder in sorted(OECD_ROOT.iterdir()):
    if not folder.is_dir():
        continue
    for parquet_file in sorted(folder.glob("*.parquet")):
        year = parquet_file.stem.split("_")[0]
        if YEARS is not None and year not in YEARS:
            continue
        print(f"OECD  {year} ...", end=" ", flush=True)
        df = pd.read_parquet(parquet_file)
        df_agg = aggregate_oecd(df)
        out_path = OUT_OECD / VERSION / folder.name / parquet_file.name
        out_path.parent.mkdir(parents=True, exist_ok=True)
        df_agg.to_parquet(out_path, engine="fastparquet", compression="gzip")
        print(f"({df.shape} -> {df_agg.shape})  saved to {out_path}")


### Diagnosis

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# OECD ICIO 2017 — diagnostic: where things live and what USA looks like
# ════════════════════════════════════════════════════════════════════════════
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Rectangle, Patch

# ── CONFIG ──────────────────────────────────────────────────────────────────
YEAR     = 2017
OECD_AGG = ROOT / "data/interim/IOT/OCDE ICIO aggregated"
FIG_DIR  = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

FD_CATS    = ["DPABR", "GFCF", "GGFC", "HFCE", "INVNT", "NPISH"]
EXTRA_ROWS = ["TLS", "VA", "OUT"]


# ── LOADING ─────────────────────────────────────────────────────────────────
def find_oecd_file(year):
    for folder in OECD_AGG.iterdir():
        if not folder.is_dir():
            continue
        for ext in ("parquet", "csv"):
            for f in folder.glob(f"{year}_*.{ext}"):
                return f
    raise FileNotFoundError(f"OECD year {year} not found")


path = find_oecd_file(YEAR)
df = pd.read_parquet(path) if path.suffix == ".parquet" \
     else pd.read_csv(path, index_col=0)
print(f"loaded: {path.name}  shape={df.shape}")


# ── CATEGORIZE ROWS AND COLUMNS ─────────────────────────────────────────────
def is_country_prefix(label):
    return "_" in label and len(label.split("_")[0]) == 3


def is_sector_label(label):
    return is_country_prefix(label) and label.split("_", 1)[1] not in FD_CATS


def is_fd_label(label):
    return is_country_prefix(label) and label.split("_", 1)[1] in FD_CATS


sector_rows = [i for i, r in enumerate(df.index)   if is_sector_label(r)]
extra_rows  = [i for i, r in enumerate(df.index)   if r in EXTRA_ROWS]
sector_cols = [j for j, c in enumerate(df.columns) if is_sector_label(c)]
fd_cols     = [j for j, c in enumerate(df.columns) if is_fd_label(c)]
out_cols    = [j for j, c in enumerate(df.columns) if c == "OUT"]

countries, sectors = [], []
seen_c, seen_s = set(), set()
for r in df.index:
    if is_sector_label(r):
        c, s = r.split("_", 1)
        if c not in seen_c:
            countries.append(c); seen_c.add(c)
        if s not in seen_s:
            sectors.append(s); seen_s.add(s)
n_c, n_sec, n_fd = len(countries), len(sectors), len(FD_CATS)

print(f"  {n_c} countries  |  {n_sec} sectors  |  {n_fd} FD categories")
print(f"  rows : {len(sector_rows)} sector rows + {len(extra_rows)} extra rows "
      f"({[df.index[i] for i in extra_rows]})")
print(f"  cols : {len(sector_cols)} sector cols + {len(fd_cols)} FD cols + "
      f"{len(out_cols)} OUT col")


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 1  —  "Map" of the full table: where each component lives
# ════════════════════════════════════════════════════════════════════════════
M = df.values.astype(float)
n_row, n_col = M.shape

# High resolution: figure size scaled so that each cell gets roughly one pixel
# (or more) at save-time DPI. With figsize=(22, 18) and dpi=300, the raster
# is ~6600 × 5400 px, well above the 3000×3500 table -> every cell is visible.
FIG1_DPI = 300
fig, ax = plt.subplots(figsize=(22, 18))
im = ax.imshow(np.clip(np.abs(M), 1, None),
               cmap="Greys",
               norm=LogNorm(vmin=1, vmax=np.abs(M).max()),
               aspect="auto",
               interpolation="nearest",
               resample=False)
cb = plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
cb.set_label("|value|  (M$, log scale)", fontsize=10)

n_row_sec = len(sector_rows)
n_col_sec = len(sector_cols)
n_col_fd  = len(fd_cols)
n_col_out = len(out_cols)

blocks = [
    # (label, row_start, col_start, height, width, color)
    ("Z  (intermediates)",
     0, 0, n_row_sec, n_col_sec, "#1f77b4"),
    ("F  (final demand)",
     0, n_col_sec, n_row_sec, n_col_fd, "#2ca02c"),
    ("OUT column",
     0, n_col_sec + n_col_fd, n_row_sec, n_col_out, "#ff7f0e"),
    ("Extra rows (TLS, VA, OUT)",
     n_row_sec, 0, len(extra_rows), n_col_sec + n_col_fd + n_col_out, "#d62728"),
]
for label, r0, c0, h, w, color in blocks:
    ax.add_patch(Rectangle((c0 - 0.5, r0 - 0.5), w, h,
                            linewidth=2.5, edgecolor=color,
                            facecolor="none", zorder=5))

# Highlight USA × USA
usa_idx_rows = [i for i in sector_rows if df.index[i].startswith("USA_")]
usa_idx_cols = [j for j in sector_cols if df.columns[j].startswith("USA_")]
if usa_idx_rows and usa_idx_cols:
    r0, r1 = usa_idx_rows[0], usa_idx_rows[-1]
    c0, c1 = usa_idx_cols[0], usa_idx_cols[-1]
    ax.add_patch(Rectangle((c0 - 0.5, r0 - 0.5), c1 - c0 + 1, r1 - r0 + 1,
                            linewidth=1.8, edgecolor="gold", facecolor="none",
                            linestyle="--", zorder=6))
    ax.text(c0 + (c1 - c0) / 2, r0 - max(20, n_row * 0.01),
            "USA × USA", ha="center", va="bottom",
            fontsize=10, color="gold", fontweight="bold")

legend = [Patch(facecolor="none", edgecolor=c, label=lbl, linewidth=2)
          for lbl, _, _, _, _, c in blocks]
legend.append(Patch(facecolor="none", edgecolor="gold", linestyle="--",
                     label="USA rows × USA sector cols"))
ax.legend(handles=legend, loc="upper left", bbox_to_anchor=(1.18, 1),
          fontsize=9, frameon=False)

ax.set_title(f"OECD ICIO {YEAR} — full table layout  ({n_row} × {n_col})",
             fontsize=14, fontweight="bold", pad=14)
ax.set_xlabel(f"Columns  →  [country × sector] | [country × FD] | OUT  "
              f"({n_col} cols)", fontsize=10)
ax.set_ylabel(f"Rows  →  [country × sector] | TLS/VA/OUT  ({n_row} rows)",
              fontsize=10)
ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig(FIG_DIR / f"oecd_{YEAR}_layout.png",
            dpi=FIG1_DPI, bbox_inches="tight")
plt.show()


# ════════════════════════════════════════════════════════════════════════════
# Extract USA-centric matrices
# ════════════════════════════════════════════════════════════════════════════
usa_sec_rows   = [r for r in df.index   if r.startswith("USA_") and is_sector_label(r)]
usa_sec_cols   = [c for c in df.columns if c.startswith("USA_") and is_sector_label(c)]
usa_fd_cols    = [c for c in df.columns if c.startswith("USA_") and is_fd_label(c)]
world_sec_rows = [r for r in df.index   if is_sector_label(r) and not r.startswith("USA_")]
world_sec_cols = [c for c in df.columns if is_sector_label(c) and not c.startswith("USA_")]
world_fd_cols  = [c for c in df.columns if is_fd_label(c)     and not c.startswith("USA_")]

# Order USA rows / cols by the sector list, for clean axes
usa_sec_rows.sort(key=lambda r: sectors.index(r.split("_", 1)[1]))
usa_sec_cols.sort(key=lambda c: sectors.index(c.split("_", 1)[1]))
usa_fd_cols.sort(key=lambda c: FD_CATS.index(c.split("_", 1)[1]))

# Intermediates
Z_us_us = df.loc[usa_sec_rows,   usa_sec_cols].values.astype(float)
Z_us_w  = df.loc[usa_sec_rows,   world_sec_cols].values.astype(float)
Z_w_us  = df.loc[world_sec_rows, usa_sec_cols].values.astype(float)
# Final demand
F_us_us = df.loc[usa_sec_rows,   usa_fd_cols].values.astype(float)
F_us_w  = df.loc[usa_sec_rows,   world_fd_cols].values.astype(float)
F_w_us  = df.loc[world_sec_rows, usa_fd_cols].values.astype(float)
# Extras (USA columns)
VA_us       = df.loc["VA",  usa_sec_cols].values.astype(float)
TLS_us      = df.loc["TLS", usa_sec_cols].values.astype(float)
OUT_us      = df.loc["OUT", usa_sec_cols].values.astype(float)
OUT_us_row  = df.loc[usa_sec_rows, "OUT"].values.astype(float)
# Trade — per USA sector (intermediates only) and per FD category (final imports)
EX_us    = Z_us_w.sum(axis=1) + F_us_w.sum(axis=1)   # per USA sector
M_us     = Z_w_us.sum(axis=0)                        # per USA sector
M_us_fd  = F_w_us.sum(axis=0)                        # per FD category


# ── Accounting checks ───────────────────────────────────────────────────────
row_supply = (Z_us_us.sum(axis=1) + Z_us_w.sum(axis=1)
              + F_us_us.sum(axis=1) + F_us_w.sum(axis=1))
row_err = OUT_us_row - row_supply

col_supply = Z_us_us.sum(axis=0) + Z_w_us.sum(axis=0) + VA_us + TLS_us
col_err = OUT_us - col_supply

print(f"\n{'─'*70}")
print(f"USA aggregates (M$, year {YEAR})")
print(f"{'─'*70}")
print(f"  Z[USA→USA]       : {Z_us_us.sum():>14,.0f}")
print(f"  Z[USA→world]     : {Z_us_w.sum():>14,.0f}   (intermediate exports)")
print(f"  Z[world→USA]     : {Z_w_us.sum():>14,.0f}   (intermediate imports)")
print(f"  F[USA, USA_FD]   : {F_us_us.sum():>14,.0f}   (domestic FD from US production)")
print(f"  F[USA, world_FD] : {F_us_w.sum():>14,.0f}   (US production to foreign FD)")
print(f"  F[world, USA_FD] : {F_w_us.sum():>14,.0f}   (FD imports)")
print(f"  VA  (USA)        : {VA_us.sum():>14,.0f}")
print(f"  TLS (USA)        : {TLS_us.sum():>14,.0f}")
print(f"  EX  (USA, total) : {EX_us.sum():>14,.0f}")
print(f"  M   intermediate : {M_us.sum():>14,.0f}")
print(f"  M   final demand : {M_us_fd.sum():>14,.0f}")
print(f"  M   total        : {M_us.sum() + M_us_fd.sum():>14,.0f}")
print(f"  OUT (USA, col)   : {OUT_us.sum():>14,.0f}")
print(f"  OUT (USA, row)   : {OUT_us_row.sum():>14,.0f}   (should match column total)")
print(f"  Row balance err  : max|err|={np.abs(row_err).max():,.1f} M$")
print(f"  Col balance err  : max|err|={np.abs(col_err).max():,.1f} M$")
print(f"{'─'*70}\n")


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 2  —  USA detail: all the extracted matrices in one shot
# ════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(20, 13))
gs  = fig.add_gridspec(3, 6, hspace=0.5, wspace=0.55,
                       height_ratios=[1.6, 1.0, 1.0])

short = [s[:12] for s in sectors]
x = np.arange(n_sec)

# 1) Z[USA × USA] — the classic IO matrix
ax = fig.add_subplot(gs[0, 0:3])
im = ax.imshow(np.clip(Z_us_us, 1, None),
               cmap="YlOrRd",
               norm=LogNorm(vmin=1, vmax=max(Z_us_us.max(), 1)),
               aspect="auto", interpolation="nearest")
ax.set_title("Z[USA × USA]  — domestic intermediate flows  (log M$)",
             fontsize=10, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(short, rotation=60, ha="right", fontsize=6)
ax.set_yticks(x); ax.set_yticklabels(short, fontsize=6)
ax.set_xlabel("Buying sector", fontsize=8)
ax.set_ylabel("Selling sector", fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.7).set_label("M$", fontsize=7)

# 2) F[USA, USA_FD] — domestic final demand
ax = fig.add_subplot(gs[0, 3:5])
F_plot = np.clip(F_us_us, 1, None)
im = ax.imshow(F_plot, cmap="PuBuGn",
               norm=LogNorm(vmin=1, vmax=max(F_plot.max(), 1)),
               aspect="auto", interpolation="nearest")
ax.set_title("F[USA, USA_FD]  — final demand by category (log M$)",
             fontsize=10, fontweight="bold")
ax.set_xticks(range(n_fd))
ax.set_xticklabels([c.split("_", 1)[1] for c in usa_fd_cols],
                    rotation=45, fontsize=7)
ax.set_yticks(x); ax.set_yticklabels(short, fontsize=6)
ax.set_xlabel("Final-demand category", fontsize=8)
ax.set_ylabel("Selling sector", fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.7).set_label("M$", fontsize=7)

# 3) Output composition by USA sector (stacked horizontal bars)
ax = fig.add_subplot(gs[0, 5])
components = np.column_stack([
    Z_us_us.sum(axis=1),                                # to domestic intermediates
    F_us_us.sum(axis=1),                                # to domestic FD
    Z_us_w.sum(axis=1) + F_us_w.sum(axis=1),            # exports
])
labels = ["→ dom. interm.", "→ dom. FD", "→ exports"]
colors_stack = ["#1f77b4", "#2ca02c", "#9467bd"]
bottom = np.zeros(n_sec)
for i, (lbl, col) in enumerate(zip(labels, colors_stack)):
    ax.barh(x, components[:, i], left=bottom, color=col, label=lbl, height=0.85)
    bottom += components[:, i]
ax.set_yticks(x); ax.set_yticklabels(short, fontsize=6)
ax.invert_yaxis()
ax.set_xlabel("M$", fontsize=8)
ax.set_title("Output decomposition by sector", fontsize=10, fontweight="bold")
ax.legend(fontsize=7, frameon=False, loc="lower right")
ax.grid(axis="x", linestyle=":", alpha=0.4)

# 4) VA per sector
ax = fig.add_subplot(gs[1, 0:2])
ax.bar(x, VA_us, color="#3577a8")
ax.set_title("Value added (VA)", fontsize=10, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(short, rotation=60, ha="right", fontsize=6)
ax.set_ylabel("M$", fontsize=8); ax.grid(axis="y", linestyle=":", alpha=0.4)

# 5) TLS per sector
ax = fig.add_subplot(gs[1, 2:4])
ax.bar(x, TLS_us, color="#bd5e2c")
ax.set_title("Taxes less subsidies (TLS)", fontsize=10, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(short, rotation=60, ha="right", fontsize=6)
ax.set_ylabel("M$", fontsize=8); ax.grid(axis="y", linestyle=":", alpha=0.4)

# 6) OUT per sector — col (supply) vs row (demand)
w = 0.4
ax = fig.add_subplot(gs[1, 4:])
ax.bar(x - w/2, OUT_us,     w, color="#777777", label="OUT (col, supply)")
ax.bar(x + w/2, OUT_us_row, w, color="#cc4c4c", label="OUT (row, demand)")
ax.set_title("Total output (column vs row)", fontsize=10, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(short, rotation=60, ha="right", fontsize=6)
ax.set_ylabel("M$", fontsize=8); ax.legend(fontsize=7, frameon=False)
ax.grid(axis="y", linestyle=":", alpha=0.4)

# 7) EX vs M (per USA sector — intermediate imports only)
ax = fig.add_subplot(gs[2, 0:3])
ax.bar(x - w/2, EX_us, w, color="#2ca02c", label="EX (exports, per sector)")
ax.bar(x + w/2, M_us,  w, color="#cc4c4c", label="M (intermediate imports, per sector)")
ax.axhline(0, color="k", linewidth=0.5)
ax.set_title("Trade per sector (USA)  — intermediate-import view", fontsize=10, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(short, rotation=60, ha="right", fontsize=6)
ax.set_ylabel("M$", fontsize=8); ax.legend(fontsize=7, frameon=False)
ax.grid(axis="y", linestyle=":", alpha=0.4)
ax.annotate(f"+ final-demand imports (not per-sector): "
            f"{M_us_fd.sum():,.0f} M$  "
            f"({', '.join(f'{FD_CATS[i]}={M_us_fd[i]:,.0f}' for i in range(n_fd))})",
            xy=(0.0, -0.30), xycoords="axes fraction",
            fontsize=7, color="0.35", style="italic")

# 8) Trade balance per sector (intermediate)
ax = fig.add_subplot(gs[2, 3:])
net = EX_us - M_us
colors_bal = ["#2ca02c" if v >= 0 else "#cc4c4c" for v in net]
ax.bar(x, net, color=colors_bal)
ax.axhline(0, color="k", linewidth=0.5)
ax.set_title("Net trade balance per sector  (EX − M_intermediate)",
             fontsize=10, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(short, rotation=60, ha="right", fontsize=6)
ax.set_ylabel("M$", fontsize=8); ax.grid(axis="y", linestyle=":", alpha=0.4)

fig.suptitle(f"OECD ICIO {YEAR} — USA detailed view  "
             f"(after extracting Z, F, VA, TLS, EX, M, OUT)",
             fontsize=13, fontweight="bold", y=0.995)
plt.savefig(FIG_DIR / f"oecd_{YEAR}_usa_detail.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"figures saved to: {FIG_DIR}")


In [ ]:
rel_row_err = np.abs(row_err) / np.maximum(np.abs(OUT_us_row), 1)
rel_col_err = np.abs(col_err) / np.maximum(np.abs(OUT_us),     1)
print(f"  Row balance err  : max {np.abs(row_err).max():,.1f} M$  "
      f"(max relative: {rel_row_err.max():.2e}, "
      f"median: {np.median(rel_row_err):.2e})")
print(f"  Col balance err  : max {np.abs(col_err).max():,.1f} M$  "
      f"(max relative: {rel_col_err.max():.2e}, "
      f"median: {np.median(rel_col_err):.2e})")


## Agregation of WiNDC

### Function

In [ ]:
# ===========================================================================
# 3. WinDC aggregation helpers
# ===========================================================================
def aggregate_windc(npz):
    """Aggregate a WiNDC IOT npz to the proposed sectors. Returns a dict of blocks.

    Mirrors the structure written by v3_construction.ipynb (run all): besides the
    core blocks (Z, F, VA, EX, M, M_interm) it carries every tax block that is
    present — the separate amounts (tax_prod, tariff, tls_int, tls_fd), the
    combined `taxes`, the OECD/SNA `TLS` (product taxes) — and the `va_convention`
    flag. After the VA convention, `VA` is gross (incl. tax_prod) and `taxes`==`TLS`.
    """
    regions  = npz["regions"].tolist()
    sectors  = npz["sectors"].tolist()
    proposed = [windc_to_proposed.get(s, s) for s in sectors]
    full = pd.MultiIndex.from_tuples([(r, p) for r in regions for p in proposed],
                                     names=["region", "proposed_sector"])

    # Z: aggregate both axes
    Z_agg = pd.DataFrame(npz["Z"], index=full, columns=full).groupby(level=["region", "proposed_sector"]).sum()
    Z_agg = Z_agg.T.groupby(level=["region", "proposed_sector"]).sum().T

    def agg_rows(arr): return pd.Series(arr, index=full).groupby(level=["region", "proposed_sector"]).sum()
    def agg_mat(arr):  return pd.DataFrame(arr, index=full).groupby(level=["region", "proposed_sector"]).sum()

    out = {"Z": Z_agg, "F": agg_mat(npz["F"])}
    # 1-D blocks: aggregate by sum, only those actually present in the table.
    for k in ("VA", "EX", "M", "M_interm", "taxes", "TLS",
              "tax_prod", "tariff", "tls_int", "tls_fd"):
        if k in npz.files:
            out[k] = agg_rows(npz[k])

    # EFFECTIVE tax RATES: rates cannot be summed -> aggregate the base (amount/rate),
    # then effective_rate = amount_agg / base_agg (per proposed sector).
    def _sdiv(a, b): return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=np.abs(b) > 1e-9)
    if all(k in npz.files for k in ("ta0", "tm0", "ty0", "tax_prod", "tariff", "tls_int", "tls_fd")):
        ys0_base = agg_rows(_sdiv(npz["tax_prod"], npz["ty0"].reshape(-1)))                  # ys0
        abs_base = agg_rows(_sdiv(npz["tls_int"] + npz["tls_fd"], npz["ta0"].reshape(-1)))   # absorption
        out["ty0"] = pd.Series(_sdiv(out["tax_prod"].values, ys0_base.values),       index=out["tax_prod"].index)
        out["tm0"] = pd.Series(_sdiv(out["tariff"].values,   out["M"].values),        index=out["tariff"].index)
        out["ta0"] = pd.Series(_sdiv((out["tls_int"] + out["tls_fd"]).values, abs_base.values), index=out["tls_int"].index)

    # scalar relabeling marker, carried through unchanged (not a sector array)
    if "va_convention" in npz.files:
        out["va_convention"] = np.asarray(npz["va_convention"])
    return out

In [ ]:
# ===========================================================================
# 5. Process WinDC IOT years  (filtered by YEARS; VERSION='' -> base dir)
# ===========================================================================
for npz_file in sorted(WINDC_ROOT.glob("IOT_*.npz")):
    year = npz_file.stem.split("_")[1]
    if YEARS is not None and year not in YEARS:
        continue
    print(f"WinDC {year} ...", end=" ", flush=True)
    npz = np.load(npz_file, allow_pickle=True)
    agg = aggregate_windc(npz)
    Z_agg = agg["Z"]

    # region / proposed-sector axes (region-major, both sorted by the groupby)
    regions_u  = list(dict.fromkeys(r for r, _ in Z_agg.index))
    sectors_u  = list(dict.fromkeys(p for _, p in Z_agg.index))

    # serialize every aggregated block as-is, so the output mirrors the input
    # structure (TLS / tax blocks / va_convention carried through when present).
    payload = {k: (v.values if hasattr(v, "values") else np.asarray(v))
               for k, v in agg.items()}
    payload.update(
        regions          = np.array(regions_u),
        proposed_sectors = np.array(sectors_u),
        index_labels     = np.array([f"{r}_{p}" for r, p in Z_agg.index]),
    )

    out_path = OUT_WINDC / VERSION / npz_file.name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(out_path, **payload)
    print(f"({npz['Z'].shape} -> {Z_agg.shape})  saved to {out_path.name}  "
          f"[{', '.join(k for k in payload if k not in ('regions','proposed_sectors','index_labels'))}]")

print("\nDone.")

### Diagnosis

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# WiNDC 2017 — diagnostic, mirrored on the OECD layout for easy comparison
# ════════════════════════════════════════════════════════════════════════════
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Rectangle, Patch

# ── CONFIG ──────────────────────────────────────────────────────────────────
YEAR      = 2017
WINDC_RAW = ROOT / f"data/interim/IOT/IOT_USA/grav_fric_{version_windc}_aggregated"
FIG_DIR   = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

WINDC_SCALE = 1000  # Bn$ -> M$ (so the units match OECD)


# ── LOADING ─────────────────────────────────────────────────────────────────
path = WINDC_RAW / f"IOT_{YEAR}.npz"
npz  = np.load(path, allow_pickle=True)
print(f"loaded: {path.name}")
print(f"keys  : {list(npz.keys())}")
for k in npz.keys():
    a = npz[k]
    print(f"  {k:20s} shape={str(tuple(a.shape)):<22s} dtype={a.dtype}")

# In raw WiNDC, `regions` may already contain flat STATE_sector labels.
# Derive the actual list of states from `index_labels`.
labels = [str(l) for l in (npz["index_labels"]
                            if "index_labels" in npz.files
                            else npz["regions"])]
regions = sorted({l.split("_")[0] for l in labels})
sectors = [str(s) for s in npz["proposed_sectors"]]
n_r, n_sec = len(regions), len(sectors)
assert n_r * n_sec == len(labels), \
    f"Inconsistent dims: {n_r} states × {n_sec} sectors ≠ {len(labels)} labels"

# Scale every monetary array Bn$ -> M$
Z_raw    = npz["Z"].astype(float)        * WINDC_SCALE   # (n_r*n_sec, n_r*n_sec)
F_raw    = npz["F"].astype(float)        * WINDC_SCALE   # (n_r*n_sec, n_fd_dim)
VA       = npz["VA"].astype(float)       * WINDC_SCALE   # (n_r*n_sec,)
EX       = npz["EX"].astype(float)       * WINDC_SCALE   # (n_r*n_sec,)
M_interm = npz["M_interm"].astype(float) * WINDC_SCALE   # (n_r*n_sec,)
taxes    = npz["taxes"].astype(float)    * WINDC_SCALE   # (n_r*n_sec,)

# Determine whether F is stored per-state×FD (n_r*n_fd) or aggregated (n_fd)
n_fd_dim = F_raw.shape[1]
if n_fd_dim >= n_r and n_fd_dim % n_r == 0:
    n_fd, F_per_state = n_fd_dim // n_r, True
else:
    n_fd, F_per_state = n_fd_dim, False

print(f"\n  {n_r} states  |  {n_sec} sectors  |  {n_fd} FD categories  "
      f"(F is {'per-state' if F_per_state else 'state-aggregated'})")

# OUT (per state×sector) — row total of the supply side
OUT = Z_raw.sum(axis=1) + F_raw.sum(axis=1) + EX  # WiNDC convention: EX is separate


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 1  —  "Layout map" as if WiNDC were stored OECD-style
# ════════════════════════════════════════════════════════════════════════════
# Reconstruct a conceptual DataFrame-like matrix:
#   rows = state×sector + TLS + VA + OUT
#   cols = state×sector + state×FD (or FD) + OUT
# EX and M_interm are 1-D vectors (no world dimension in WiNDC), so they are
# shown as extra annotated stripes on the right, not as a column of the matrix.

n_row_sec = n_r * n_sec
fd_n      = n_r * n_fd if F_per_state else n_fd
extra_rows_names = ["TLS", "VA", "OUT"]
extra_rows_data  = [taxes, VA, OUT]

# Extra "stripe" columns: EX and M_interm — shown on the right of the matrix
stripe_names = ["EX", "M_interm"]
stripe_data  = [EX, M_interm]

n_full_rows = n_row_sec + len(extra_rows_names)
n_full_cols = n_row_sec + fd_n + 1 + len(stripe_names)  # +1 for OUT col

M_full = np.zeros((n_full_rows, n_full_cols))
M_full[:n_row_sec, :n_row_sec]                 = Z_raw
M_full[:n_row_sec, n_row_sec:n_row_sec + fd_n] = F_raw
M_full[:n_row_sec, n_row_sec + fd_n]           = OUT
for k, vec in enumerate(stripe_data):
    M_full[:n_row_sec, n_row_sec + fd_n + 1 + k] = vec
for k, vec in enumerate(extra_rows_data):
    M_full[n_row_sec + k, :n_row_sec] = vec

print(f"\nReconstructed WiNDC IO-style matrix: {M_full.shape}")

fig, ax = plt.subplots(figsize=(22, 18))
im = ax.imshow(np.clip(np.abs(M_full), 1, None),
               cmap="Greys",
               norm=LogNorm(vmin=1, vmax=max(np.abs(M_full).max(), 10)),
               aspect="auto",
               interpolation="nearest",
               resample=False)
cb = plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
cb.set_label("|value|  (M$, log scale)", fontsize=10)

# Color-coded zones (same palette as the OECD figure)
blocks = [
    ("Z  (intermediates)",
     0, 0, n_row_sec, n_row_sec, "#1f77b4"),
    ("F  (final demand)",
     0, n_row_sec, n_row_sec, fd_n, "#2ca02c"),
    ("OUT column",
     0, n_row_sec + fd_n, n_row_sec, 1, "#ff7f0e"),
    ("EX / M_interm (1-D stripes — no world dim in WiNDC)",
     0, n_row_sec + fd_n + 1, n_row_sec, len(stripe_names), "#9467bd"),
    ("Extra rows (TLS, VA, OUT)",
     n_row_sec, 0, len(extra_rows_names), n_full_cols, "#d62728"),
]
for label, r0, c0, h, w, color in blocks:
    ax.add_patch(Rectangle((c0 - 0.5, r0 - 0.5), w, h,
                            linewidth=2.5, edgecolor=color,
                            facecolor="none", zorder=5))

# Highlight one state's intra-block (analogue of OECD's USA×USA gold box)
hl_state = sorted(regions)[0]
hl_idx   = regions.index(hl_state)
r0 = hl_idx * n_sec; r1 = r0 + n_sec - 1
c0 = hl_idx * n_sec; c1 = c0 + n_sec - 1
ax.add_patch(Rectangle((c0 - 0.5, r0 - 0.5), c1 - c0 + 1, r1 - r0 + 1,
                        linewidth=1.8, edgecolor="gold", facecolor="none",
                        linestyle="--", zorder=6))
ax.text(c0 + (c1 - c0) / 2, r0 - max(15, n_full_rows * 0.01),
        f"{hl_state} × {hl_state}", ha="center", va="bottom",
        fontsize=10, color="gold", fontweight="bold")

legend = [Patch(facecolor="none", edgecolor=c, label=lbl, linewidth=2)
          for lbl, _, _, _, _, c in blocks]
legend.append(Patch(facecolor="none", edgecolor="gold", linestyle="--",
                     label=f"{hl_state} rows × {hl_state} sector cols"))
ax.legend(handles=legend, loc="upper left", bbox_to_anchor=(1.18, 1),
          fontsize=9, frameon=False)

ax.set_title(f"WiNDC {YEAR} — reconstructed IO-style layout  "
             f"({M_full.shape[0]} × {M_full.shape[1]})\n"
             f"(values rescaled Bn$ → M$ for comparability with OECD)",
             fontsize=14, fontweight="bold", pad=14)
ax.set_xlabel(f"Columns  →  [state × sector] | [state × FD] | OUT | EX | M_interm  "
              f"({M_full.shape[1]} cols)", fontsize=10)
ax.set_ylabel(f"Rows  →  [state × sector] | TLS/VA/OUT  ({M_full.shape[0]} rows)",
              fontsize=10)
ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig(FIG_DIR / f"windc_{YEAR}_layout.png", dpi=300, bbox_inches="tight")
plt.show()


# ════════════════════════════════════════════════════════════════════════════
# Aggregate WiNDC to USA level (sum over states) — same axes as OECD detail
# ════════════════════════════════════════════════════════════════════════════
Z_us_us = Z_raw.reshape(n_r, n_sec, n_r, n_sec).sum(axis=(0, 2))   # (n_sec, n_sec)
if F_per_state:
    F_us_us = F_raw.reshape(n_r, n_sec, n_r, n_fd).sum(axis=(0, 2))   # (n_sec, n_fd)
else:
    F_us_us = F_raw.reshape(n_r, n_sec, n_fd).sum(axis=0)             # (n_sec, n_fd)

VA_us       = VA.reshape(n_r, n_sec).sum(axis=0)
TLS_us      = taxes.reshape(n_r, n_sec).sum(axis=0)
EX_us       = EX.reshape(n_r, n_sec).sum(axis=0)
M_us        = M_interm.reshape(n_r, n_sec).sum(axis=0)
OUT_us_row  = (Z_us_us.sum(axis=1) + F_us_us.sum(axis=1) + EX_us)
OUT_us_col  = (Z_us_us.sum(axis=0) + VA_us + TLS_us + M_us)


# Accounting checks (row vs column total per sector)
row_col_err = OUT_us_row - OUT_us_col
rel_err     = np.abs(row_col_err) / np.maximum(np.abs(OUT_us_row), 1)

print(f"\n{'─'*70}")
print(f"USA aggregates (WiNDC raw, scaled to M$, year {YEAR})")
print(f"{'─'*70}")
print(f"  Z[USA→USA]         : {Z_us_us.sum():>14,.0f}")
print(f"  F[USA, USA_FD]     : {F_us_us.sum():>14,.0f}")
print(f"  VA  (USA)          : {VA_us.sum():>14,.0f}")
print(f"  TLS (USA, taxes)   : {TLS_us.sum():>14,.0f}")
print(f"  EX  (USA, exports) : {EX_us.sum():>14,.0f}")
print(f"  M   (USA, interm.) : {M_us.sum():>14,.0f}")
print(f"  OUT (row, supply)  : {OUT_us_row.sum():>14,.0f}")
print(f"  OUT (col, demand)  : {OUT_us_col.sum():>14,.0f}")
print(f"  Row-col balance err: max {np.abs(row_col_err).max():,.1f} M$  "
      f"\n(max rel: {rel_err.max():.2e}, median rel: {np.median(rel_err):.2e})")
print(f"{'─'*70}\n")


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 2  —  USA detail (8 panels, same layout as the OECD diagnostic)
# ════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(20, 13))
gs  = fig.add_gridspec(3, 6, hspace=0.5, wspace=0.55,
                       height_ratios=[1.6, 1.0, 1.0])

short = [s[:12] for s in sectors]
x = np.arange(n_sec)

# Try to label FD axes if WiNDC stored names; otherwise generic labels
fd_labels = (list(npz["fd_categories"]) if "fd_categories" in npz.files
             else [f"FD_{i+1}" for i in range(n_fd)])

# 1) Z[USA × USA] — domestic intermediate flows
ax = fig.add_subplot(gs[0, 0:3])
im = ax.imshow(np.clip(Z_us_us, 1, None), cmap="YlOrRd",
               norm=LogNorm(vmin=1, vmax=max(Z_us_us.max(), 1)),
               aspect="auto", interpolation="nearest")
ax.set_title("Z[USA × USA]  — domestic intermediate flows (log M$)",
             fontsize=10, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(short, rotation=60, ha="right", fontsize=6)
ax.set_yticks(x); ax.set_yticklabels(short, fontsize=6)
ax.set_xlabel("Buying sector",  fontsize=8)
ax.set_ylabel("Selling sector", fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.7).set_label("M$", fontsize=7)

# 2) F[USA, USA_FD] — domestic final demand
ax = fig.add_subplot(gs[0, 3:5])
F_plot = np.clip(F_us_us, 1, None)
im = ax.imshow(F_plot, cmap="PuBuGn",
               norm=LogNorm(vmin=1, vmax=max(F_plot.max(), 1)),
               aspect="auto", interpolation="nearest")
ax.set_title("F[USA, USA_FD]  — final demand by category (log M$)",
             fontsize=10, fontweight="bold")
ax.set_xticks(range(n_fd))
ax.set_xticklabels(fd_labels, rotation=45, fontsize=7)
ax.set_yticks(x); ax.set_yticklabels(short, fontsize=6)
ax.set_xlabel("Final-demand category", fontsize=8)
ax.set_ylabel("Selling sector",         fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.7).set_label("M$", fontsize=7)

# 3) Output composition by USA sector
ax = fig.add_subplot(gs[0, 5])
components = np.column_stack([
    Z_us_us.sum(axis=1),       # → domestic intermediates
    F_us_us.sum(axis=1),       # → domestic FD
    EX_us,                     # → exports (rest of world)
])
labels = ["→ dom. interm.", "→ dom. FD", "→ exports"]
colors_stack = ["#1f77b4", "#2ca02c", "#9467bd"]
bottom = np.zeros(n_sec)
for i, (lbl, col) in enumerate(zip(labels, colors_stack)):
    ax.barh(x, components[:, i], left=bottom, color=col, label=lbl, height=0.85)
    bottom += components[:, i]
ax.set_yticks(x); ax.set_yticklabels(short, fontsize=6)
ax.invert_yaxis()
ax.set_xlabel("M$", fontsize=8)
ax.set_title("Output decomposition by sector", fontsize=10, fontweight="bold")
ax.legend(fontsize=7, frameon=False, loc="lower right")
ax.grid(axis="x", linestyle=":", alpha=0.4)

# 4) VA per sector
ax = fig.add_subplot(gs[1, 0:2])
ax.bar(x, VA_us, color="#3577a8")
ax.set_title("Value added (VA)", fontsize=10, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(short, rotation=60, ha="right", fontsize=6)
ax.set_ylabel("M$", fontsize=8); ax.grid(axis="y", linestyle=":", alpha=0.4)

# 5) TLS per sector
ax = fig.add_subplot(gs[1, 2:4])
ax.bar(x, TLS_us, color="#bd5e2c")
ax.set_title("Taxes less subsidies (TLS)", fontsize=10, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(short, rotation=60, ha="right", fontsize=6)
ax.set_ylabel("M$", fontsize=8); ax.grid(axis="y", linestyle=":", alpha=0.4)

# 6) OUT per sector — row vs column
w = 0.4
ax = fig.add_subplot(gs[1, 4:])
ax.bar(x - w/2, OUT_us_col, w, color="#777777", label="OUT (col, supply)")
ax.bar(x + w/2, OUT_us_row, w, color="#cc4c4c", label="OUT (row, demand)")
ax.set_title("Total output (column vs row)", fontsize=10, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(short, rotation=60, ha="right", fontsize=6)
ax.set_ylabel("M$", fontsize=8); ax.legend(fontsize=7, frameon=False)
ax.grid(axis="y", linestyle=":", alpha=0.4)

# 7) EX vs M per USA sector
ax = fig.add_subplot(gs[2, 0:3])
ax.bar(x - w/2, EX_us, w, color="#2ca02c", label="EX (exports)")
ax.bar(x + w/2, M_us,  w, color="#cc4c4c", label="M (intermediate imports)")
ax.axhline(0, color="k", linewidth=0.5)
ax.set_title("Trade per sector (USA)", fontsize=10, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(short, rotation=60, ha="right", fontsize=6)
ax.set_ylabel("M$", fontsize=8); ax.legend(fontsize=7, frameon=False)
ax.grid(axis="y", linestyle=":", alpha=0.4)

# 8) Net trade balance per sector
ax = fig.add_subplot(gs[2, 3:])
net = EX_us - M_us
colors_bal = ["#2ca02c" if v >= 0 else "#cc4c4c" for v in net]
ax.bar(x, net, color=colors_bal)
ax.axhline(0, color="k", linewidth=0.5)
ax.set_title("Net trade balance per sector  (EX − M)",
             fontsize=10, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(short, rotation=60, ha="right", fontsize=6)
ax.set_ylabel("M$", fontsize=8); ax.grid(axis="y", linestyle=":", alpha=0.4)

fig.suptitle(f"WiNDC {YEAR} — USA detailed view  "
             f"(state-aggregated, rescaled to M$)",
             fontsize=13, fontweight="bold", y=0.995)
plt.savefig(FIG_DIR / f"windc_{YEAR}_usa_detail.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"figures saved to: {FIG_DIR}")


In [ ]:
gdp_income      = VA_us.sum() + TLS_us.sum()
gdp_expenditure = F_us_us.sum() + EX_us.sum() - M_us.sum()
gap = gdp_income - gdp_expenditure
print(f"GDP (income side)      : {gdp_income:>15,.0f} M$")
print(f"GDP (expenditure side) : {gdp_expenditure:>15,.0f} M$")
print(f"Gap                    : {gap:>+15,.0f} M$  ({gap/gdp_income:+.2%})")
